In [3]:
%matplotlib inline
%matplotlib widget
"""
PINN PDE Solver - 训练演示脚本
"""

import sys
import torch

# ===== 1. 设置路径 =====
project_root = '..'
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# ===== 2. 导入 src 包 =====
from src import *
# ===== 3. 定义问题配置 =====
# 可选: "1d_steady", "1d_transient", "2d_steady", "2d_transient"
PROBLEM_NAME = "2d_transient"
PROBLEMS = {
    "1d_steady": {
        "dimension": 1, "order": 2, "has_t": False,
        "coeffs": [1, 0, 1],
        "source_term": "0",
        "domain": {"x": [0, 1]},
        "condition": [
            {"point": 0.0, "value": 1.0, "derivative": 0},
            {"point": 0.0, "value": 0.0, "derivative": 1},
        ],
    },
    "1d_transient": {
        "dimension": 1, "order": 2, "has_t": True,
        "coeffs": {"u_t": 1.0, "u_xx": -0.1},
        "source_term": "0",
        "domain": {"x": [0, 1], "t": [0, 0.5]},
        "condition": [
            {"side": "initial", "derivative": 0, "value": "sin(pi*x)"},
            {"side": "left", "type": "dirichlet", "value": "0"},
            {"side": "right", "type": "dirichlet", "value": "0"},
        ],
    },
    "2d_steady": {
        "dimension": 2, "order": 2, "has_t": False,
        "coeffs": {"u_xx": 1.0, "u_yy": 1.0},
        "source_term": "sin(pi*x)*sin(pi*y)",
        "domain": {"x": [0, 1], "y": [0, 1]},
        "condition": [
            {"side": "left", "type": "dirichlet", "value": "0"},
            {"side": "right", "type": "dirichlet", "value": "0"},
            {"side": "bottom", "type": "dirichlet", "value": "0"},
            {"side": "top", "type": "dirichlet", "value": "0"},
        ],
    },
    "2d_transient": {
        "dimension": 2, "order": 2, "has_t": True,
        "coeffs": {"u_t": 1.0, "u_xx": 0.1, "u_yy": 0.1},
        "source_term": "0",
        "domain": {"x": [0, 1], "y": [0, 1], "t": [0, 0.5]},
        "condition": [
            {"side": "initial", "derivative": 0, "value": "sin(pi*x)*sin(pi*y)"},
            {"side": "left", "type": "dirichlet", "value": "0"},
            {"side": "right", "type": "dirichlet", "value": "0"},
            {"side": "bottom", "type": "dirichlet", "value": "0"},
            {"side": "top", "type": "dirichlet", "value": "0"},
        ],
    },
}
problem = PROBLEMS[PROBLEM_NAME]
print(f"📋 当前问题: {PROBLEM_NAME}")
# ===== 4. 生成损失函数 =====
def build_loss_functions(problem):
    dimension = problem["dimension"]
    has_t = problem["has_t"]
    coeffs = problem["coeffs"]
    source_term = problem["source_term"]
    condition = problem["condition"]
    order = problem["order"]
    if dimension == 1 and not has_t:
        coeff_funcs = InputParser.parse_coeffs_1d(coeffs, ["x"])
        f, _ = InputParser.parse_source(source_term, ["x"])
        return LossGenerator.generate(
            dimension=1, has_t=False,
            coeff_funcs=coeff_funcs, f=f,
            condition=condition, order=order
        )
    elif dimension == 1 and has_t:
        parsed = InputParser.parse_coeffs_1d_transient(coeffs, ["x", "t"])
        f, _ = InputParser.parse_source(source_term, ["x", "t"])
        structured = InputParser.parse_conditions(condition, ["x", "t"])
        ic_conds = structured["initial"]
        bc_sides = {"left": [], "right": []}
        for bc in structured["boundary"]:
            side = bc.get("location_clean")
            if side in bc_sides:
                bc_sides[side].append(bc)
        return LossGenerator.generate(
            dimension=1, has_t=True,
            c_tt_fn=parsed.get("u_tt", lambda x, t: 0.0),
            c_t_fn=parsed.get("u_t", lambda x, t: 0.0),
            c_xx_fn=parsed.get("u_xx", lambda x, t: 0.0),
            c_x_fn=parsed.get("u_x", lambda x, t: 0.0),
            c_u_fn=parsed.get("u", lambda x, t: 0.0),
            f=f, ic_conds=ic_conds, bc_sides=bc_sides
        )
    elif dimension == 2 and not has_t:
        parsed = InputParser.parse_coeffs_2d(coeffs, ["x", "y"])
        f, _ = InputParser.parse_source(source_term, ["x", "y"])
        structured = InputParser.parse_conditions(condition, ["x", "y"])
        bc_sides = {"left": [], "right": [], "bottom": [], "top": []}
        for bc in structured["boundary"]:
            side = bc.get("location_clean")
            if side in bc_sides:
                bc_sides[side].append(bc)
        return LossGenerator.generate(
            dimension=2, has_t=False,
            c_xx_fn=parsed.get("u_xx", lambda x, y: 0.0),
            c_yy_fn=parsed.get("u_yy", lambda x, y: 0.0),
            c_xy_fn=parsed.get("u_xy", lambda x, y: 0.0),
            c_x_fn=parsed.get("u_x", lambda x, y: 0.0),
            c_y_fn=parsed.get("u_y", lambda x, y: 0.0),
            c_u_fn=parsed.get("u", lambda x, y: 0.0),
            f=f, bc_sides=bc_sides
        )
    elif dimension == 2 and has_t:
        parsed = InputParser.parse_coeffs_2d_transient(coeffs, ["x", "y", "t"])
        f, _ = InputParser.parse_source(source_term, ["x", "y", "t"])
        structured = InputParser.parse_conditions(condition, ["x", "y", "t"])
        ic_conds = structured["initial"]
        bc_sides = {"left": [], "right": [], "bottom": [], "top": []}
        for bc in structured["boundary"]:
            side = bc.get("location_clean")
            if side in bc_sides:
                bc_sides[side].append(bc)
        return LossGenerator.generate(
            dimension=2, has_t=True,
            c_tt_fn=parsed.get("u_tt", lambda x, y, t: 0.0),
            c_t_fn=parsed.get("u_t", lambda x, y, t: 0.0),
            c_xx_fn=parsed.get("u_xx", lambda x, y, t: 0.0),
            c_yy_fn=parsed.get("u_yy", lambda x, y, t: 0.0),
            c_xy_fn=parsed.get("u_xy", lambda x, y, t: 0.0),
            c_x_fn=parsed.get("u_x", lambda x, y, t: 0.0),
            c_y_fn=parsed.get("u_y", lambda x, y, t: 0.0),
            c_u_fn=parsed.get("u", lambda x, y, t: 0.0),
            f=f, ic_conds=ic_conds, bc_sides=bc_sides
        )
    else:
        raise ValueError(f"不支持的配置")
loss_functions = build_loss_functions(problem)
print("✅ 损失函数生成完成")
# ===== 5. 构建模型 =====
model = build_model(
    coeffs=problem["coeffs"],
    source_term=problem["source_term"],
    conditions=problem["condition"],
    has_t=problem["has_t"],
    dimension=problem["dimension"],
    verbose=True,
)
print(f"✅ 模型构建完成，参数量: {sum(p.numel() for p in model.parameters()):,}")
# ===== 6. 创建采样器 =====
domain = problem["domain"]
sampler = DomainSampler(domain["x"], domain.get("y"), domain.get("t"))
print(f"✅ 采样器创建完成，维度: {sampler.dim}D")
# ===== 7. 创建训练器 =====
trainer = PINNTrainer(
    model=model,
    loss_functions=loss_functions,
    optimizer="adam",
    lr=1e-3,
    scheduler="plateau",
    device="cpu",
)
print("✅ 训练器创建完成")
# ===== 8. 训练 =====
print(f"\n🚀 开始训练...")
print("-" * 60)
history = trainer.train(
    n_epochs=50,
    sampler=sampler,
    n_interior=1000,
    n_boundary_per_side=50,
    n_initial=200,
    verbose=True,
    eval_interval=300,
)
print("-" * 60)
print(f"✅ 训练完成！最终损失: {history['total_loss'][-1]:.3e}")
# ===== 9. 可视化 =====
print("\n📊 生成可视化...")
# 获取精确解
result = solve_pde(
    dimension=problem["dimension"],
    order=problem["order"],
    has_t=problem["has_t"],
    coeffs=problem["coeffs"],
    source_term=problem["source_term"],
    domain=problem["domain"],
    condition=problem["condition"],
)
exact_func = result["exact_solution"]
# 1D 稳态
if problem["dimension"] == 1 and not problem["has_t"]:
    def true_func(x):
        if exact_func is not None:
            return torch.tensor([exact_func(xi.item()) for xi in x], dtype=torch.float32).reshape(-1, 1)
        return None
    plot_1d_solution(model, (0, 1), 200, true_func, "PINN vs Exact")
elif problem["dimension"] == 1 and problem["has_t"]:
    t_range = problem['domain']['t']
    plot_1d_transient_interactive(
        model, (0, 1), t_range, 200, exact_func, f"{PROBLEM_NAME}"
    )
# 2D 稳态
elif problem["dimension"] == 2 and not problem["has_t"]:
    def true_func_2d(pts):
        if exact_func is not None:
            return torch.tensor([exact_func(xi.item(), yi.item()) for xi, yi in pts], dtype=torch.float32).reshape(-1, 1)
        return None
    plot_2d_solution(model, (0, 1), (0, 1), 100, true_func_2d, "2D Solution")
elif problem["dimension"] == 2 and problem["has_t"]:
    t_range = problem['domain']['t']
    plot_2d_transient_interactive(
        model, (0, 1), (0, 1), t_range, 80, exact_func, f"{PROBLEM_NAME}"
    )
print("\n✅ 全部完成！")



📋 当前问题: 2d_transient
✅ 损失函数生成完成

[NetworkFactory] --- 智能化物理场审计中 ---
[NetworkFactory] 最终完备复杂度总评分: 32.80 (维度: 2D, 含时: True)
[NetworkFactory] 推荐模型拓扑结构: [64, 64, 64]
[NetworkFactory] 适配激活函数类型: 'tanh'
✅ 模型构建完成，参数量: 8,641
✅ 采样器创建完成，维度: 3D
✅ 训练器创建完成

🚀 开始训练...
------------------------------------------------------------


  2%|▏         | 1/50 [00:00<00:19,  2.53it/s]

Epoch     0/50 | Loss: 4.191e-01 | PDE: 8.724e-02 | BC: 3.318e-01 | LR: 1.00e-03


100%|██████████| 50/50 [00:19<00:00,  2.62it/s]

Epoch    49/50 | Loss: 2.220e-01 | PDE: 6.956e-03 | BC: 2.150e-01 | LR: 1.00e-03
训练完成
------------------------------------------------------------
✅ 训练完成！最终损失: 2.220e-01

📊 生成可视化...


interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Time (t):', layout=Layout(w…


✅ 全部完成！


In [1]:
# 以下都是测试代码块
import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ---------- 模拟模型 ----------
class MockTransientModel1D:
    def eval(self): pass
    def __call__(self, x):
        if x.shape[1] == 2:
            return torch.sin(np.pi * x[:, 0:1]) * torch.exp(-x[:, 1:2])
        return torch.zeros_like(x[:, 0:1])

def true_1d(x, t):
    return np.sin(np.pi * x) * np.exp(-t)

model = MockTransientModel1D()

# ---------- 定义绘图函数（不含 plt.close()） ----------
def plot_1d_transient_interactive_no_close(
    model, x_range=(0, 1), t_range=(0, 1), n_points=200,
    exact_func=None, title="1D Transient Solution", save_path=None
):
    x_min, x_max = x_range
    t_min, t_max = t_range
    
    def _update_plot(t_val):
        x_test = torch.linspace(x_min, x_max, n_points).reshape(-1, 1)
        t_vals = torch.full((n_points, 1), t_val)
        xt = torch.cat([x_test, t_vals], dim=1)
        model.eval()
        with torch.no_grad():
            u_pred = model(xt).numpy().flatten()
        
        plt.figure(figsize=(8, 5))
        plt.plot(x_test.numpy().flatten(), u_pred, 'b-', label='PINN Prediction', linewidth=2)
        if exact_func is not None:
            u_true = np.array([exact_func(xi.item(), t_val) for xi in x_test])
            plt.plot(x_test.numpy().flatten(), u_true, 'r--', label='Exact Solution', linewidth=2)
            error = np.abs(u_pred - u_true)
            plt.text(0.05, 0.95, f"Max error: {error.max():.2e}\nMean error: {error.mean():.2e}",
                     transform=plt.gca().transAxes, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        plt.xlabel('x')
        plt.ylabel(f'u(x, t={t_val:.3f})')
        plt.title(f'{title} (t={t_val:.3f})')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xlim(x_min, x_max)
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
    
    slider = widgets.FloatSlider(
        value=(t_min + t_max) / 2,
        min=t_min,
        max=t_max,
        step=0.01,
        description='t:',
        continuous_update=True,
        style={'description_width': 'initial'}
    )
    display(widgets.interactive(_update_plot, t_val=slider))

# ---------- 运行测试 ----------
print("测试：移除 plt.close() 的版本")
plot_1d_transient_interactive_no_close(
    model,
    x_range=(0, 1),
    t_range=(0, 1),
    n_points=200,
    exact_func=true_1d,
    title="Test: u(x,t) = sin(πx)·e^(-t)"
)

测试：移除 plt.close() 的版本


interactive(children=(FloatSlider(value=0.5, description='t:', max=1.0, step=0.01, style=SliderStyle(descripti…